# Сравнительный анализ подходов к векторизации текста

**Цель:** сравнить логистическую регрессию на четырёх представлениях текста:
TF-IDF по словам, TF-IDF по словам и биграммам, усреднённые GloVe и `[CLS]` DistilBERT.

**Данные:** первые 2000 отзывов SST-2, классы `0` — отрицательный, `1` — положительный.
Все методы используют одни и те же 1500 обучающих и 500 тестовых примеров.
В этой работе обучается только логистическая регрессия; веса GloVe и DistilBERT фиксированы.

Запускайте ячейки по порядку, выбрав Python из `venv` этой папки.
Файл `lab.py` рядом с ноутбуком содержит загрузку, кэширование и сохранение отчёта,
а также позволяет повторить всю работу командой `./venv/bin/python lab.py`.

## Задание 1. Подготовка среды и данных

In [1]:
import gc
import re
import lab  # Настраивает кэш и потоки до импорта численных библиотек.
import numpy as np
import pandas as pd
import torch
import transformers as ppb
import gensim.downloader
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from IPython.display import display, Markdown

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)
records = []
print("Устройство: CPU; random_state=42; потоки PyTorch: 2")

/home/salamat/Downloads/labka5/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: CPU; random_state=42; потоки PyTorch: 2


In [2]:
# Загрузчик скачивает указанный в задании TSV только при отсутствии локального файла.
# Внутри: pd.read_csv(path, sep="\t", header=None).
df = lab.load_data()
batch_1 = df.iloc[:2000].copy()
print(f"Полный датасет: {len(df)}; используем: {len(batch_1)}")
display(batch_1.head().rename(columns={0: "Текст", 1: "Класс"}))

class_balance = pd.DataFrame({
    "Количество": batch_1[1].value_counts().sort_index(),
    "Доля": batch_1[1].value_counts(normalize=True).sort_index(),
})
class_balance.index.name = "Класс"
display(class_balance)
assert len(batch_1) == 2000 and set(batch_1[1].unique()) == {0, 1}

Полный датасет: 6920; используем: 2000


,Текст,Класс
0,"a stirring , funny and finally transporting re...",1
1,apparently reassembled from the cutting room f...,0
2,they presume their audience wo n't sit still f...,0
3,this is a visually stunning rumination on love...,1
4,jonathan parker 's bartleby should have been t...,1


,Количество,Доля
Класс,,
0,959,0.4795
1,1041,0.5205


## Задание 2. TF-IDF и N-граммы

Разбиение делаем один раз по индексам, чтобы при переходе от текстов к векторам
каждый отзыв оставался в своей части выборки. `test_size=0.25`, `random_state=42`.
Параметр `stratify` не добавляем, чтобы сохранить разбиение из условия.
Словарь и IDF обучаем **только на train**: использование test при `fit` создало бы утечку данных.

In [3]:
train_idx, test_idx = train_test_split(
    np.arange(len(batch_1)), test_size=0.25, random_state=42
)
X_train = batch_1[0].iloc[train_idx]
X_test = batch_1[0].iloc[test_idx]
y_train = batch_1[1].iloc[train_idx]
y_test = batch_1[1].iloc[test_idx]
print(f"Train: {len(X_train)}, test: {len(X_test)}")
assert not set(train_idx) & set(test_idx)

# Базовый ориентир: всегда предсказывать частый класс, определённый по train.
majority_class = int(y_train.mode().iloc[0])
baseline = accuracy_score(y_test, np.full(len(y_test), majority_class))
print(f"Majority baseline на test: {baseline:.4f}")

def remember_result(name, model, train_vectors, test_vectors):
    train_acc = accuracy_score(y_train, model.predict(train_vectors))
    test_acc = accuracy_score(y_test, model.predict(test_vectors))
    record = {
        "method": name,
        "train_accuracy": float(train_acc),
        "test_accuracy": float(test_acc),
        "gap": float(train_acc - test_acc),
        "n_features": int(train_vectors.shape[1]),
    }
    # Повтор ячейки обновляет строку, а не добавляет дубликат.
    records[:] = [r for r in records if r["method"] != name]
    records.append(record)
    print(f"{name} Train Accuracy: {train_acc:.4f}")
    print(f"{name} Test Accuracy:  {test_acc:.4f}")
    print(f"Количество признаков: {train_vectors.shape[1]}")

Train: 1500, test: 500
Majority baseline на test: 0.5080


### 2.1. TF-IDF (Unigrams)

In [4]:
tfidf = TfidfVectorizer(min_df=5, ngram_range=(1, 1))
Xtrain = tfidf.fit_transform(X_train)
Xtest = tfidf.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=2000, random_state=42)
lr_tfidf.fit(Xtrain, y_train)
remember_result("TF-IDF (Unigrams)", lr_tfidf, Xtrain, Xtest)

TF-IDF (Unigrams) Train Accuracy: 0.8327
TF-IDF (Unigrams) Test Accuracy:  0.6740
Количество признаков: 709


### 2.2. TF-IDF (N-Grams)

Диапазон `(1, 2)` сохраняет отдельные слова и добавляет пары соседних слов.
Это позволяет различать выражения вроде `not good` и `good`, если они прошли порог
`min_df=5`. На маленькой выборке редкие сочетания отбрасываются, поэтому улучшение не гарантировано.

In [5]:
tfidf_ngrams = TfidfVectorizer(min_df=5, ngram_range=(1, 2))
Xtrain_ngrams = tfidf_ngrams.fit_transform(X_train)
Xtest_ngrams = tfidf_ngrams.transform(X_test)

lr_ngrams = LogisticRegression(max_iter=2000, random_state=42)
lr_ngrams.fit(Xtrain_ngrams, y_train)
remember_result("TF-IDF (N-Grams)", lr_ngrams, Xtrain_ngrams, Xtest_ngrams)

TF-IDF (N-Grams) Train Accuracy: 0.8500
TF-IDF (N-Grams) Test Accuracy:  0.6660
Количество признаков: 1002


## Задание 3. Усреднённые эмбеддинги GloVe

Используем `gensim.downloader.load('glove-twitter-25')`.
Приводим текст к нижнему регистру, выделяем слова регулярным выражением,
пропускаем слова вне словаря и усредняем векторы известных слов.
Если известных слов нет, возвращаем нулевой вектор длины 25.
Порядок слов при усреднении теряется; эти векторы предобучены на Twitter, а здесь отзывы о кино.

In [6]:
glove_vectors = lab.load_glove()  # Внутри gensim.downloader.load('glove-twitter-25').
print("Размер словаря:", len(glove_vectors))
print("Размерность слова:", glove_vectors.vector_size)

def get_sentence_embedding(sentence):
    words = re.findall(lab.TOKEN_PATTERN.pattern, sentence.lower())
    vectors = [glove_vectors[word] for word in words if word in glove_vectors]
    if not vectors:
        return np.zeros(glove_vectors.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

glove_features = np.vstack(batch_1[0].map(get_sentence_embedding))
assert glove_features.shape == (2000, 25)
assert np.isfinite(glove_features).all()
np.testing.assert_array_equal(get_sentence_embedding(""), np.zeros(25))
print("Размер матрицы GloVe:", glove_features.shape)
print("Текстов без известных слов:", np.count_nonzero(~glove_features.any(axis=1)))

lr_glove = LogisticRegression(max_iter=2000, random_state=42)
lr_glove.fit(glove_features[train_idx], y_train)
remember_result("GloVe (Averaged)", lr_glove,
                glove_features[train_idx], glove_features[test_idx])

# Большой словарь больше не нужен; освобождаем память перед DistilBERT.
del glove_vectors
_ = gc.collect()

Загрузка glove-twitter-25 (при первом запуске потребуется скачивание)…


Размер словаря: 1193514
Размерность слова: 25
Размер матрицы GloVe: (2000, 25)
Текстов без известных слов: 0
GloVe (Averaged) Train Accuracy: 0.7073
GloVe (Averaged) Test Accuracy:  0.7140
Количество признаков: 25


## Задание 4. Признаки предобученного DistilBERT

### 4.1. Модель и токенизатор

Загружаем именно базовую `distilbert-base-uncased`, без готовой головы классификации
и без дообучения на SST-2. `eval()` отключает dropout.
Функция загрузки сначала проверяет локальный кэш, затем при необходимости скачивает модель.

In [7]:
model_class, tokenizer_class, pretrained_weights = (
    ppb.DistilBertModel, ppb.DistilBertTokenizer, "distilbert-base-uncased"
)
tokenizer, model = lab.load_distilbert(device="cpu")
model.eval()
print("Модель:", type(model).__name__)
print("Токенизатор:", type(tokenizer).__name__)
print("Размер скрытого состояния:", model.config.dim)

Загрузка distilbert-base-uncased на cpu…


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14170.42it/s]


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: DistilBertModel
Токенизатор: DistilBertTokenizer
Размер скрытого состояния: 768


### 4.2. Токенизация, паддинг и маска

`encode(..., add_special_tokens=True)` добавляет `[CLS]` в начало и `[SEP]` в конец.
Каждую последовательность дополняем нулями до максимальной длины в выборке.
`attention_mask` содержит 1 для настоящих токенов (включая специальные) и 0 для паддинга.
Ограничение в 512 токенов защищает от превышения длины входа модели.

In [8]:
tokenized = batch_1[0].map(
    lambda text: tokenizer.encode(text, add_special_tokens=True,
                                  truncation=True, max_length=512)
)
max_len = max(map(len, tokenized))
padded = np.array([
    ids + [tokenizer.pad_token_id] * (max_len - len(ids)) for ids in tokenized
], dtype=np.int64)
attention_mask = (padded != tokenizer.pad_token_id).astype(np.int64)

print("Максимальная длина:", max_len)
print("Форма padded:", padded.shape)
print("Первые токены:", tokenizer.convert_ids_to_tokens(tokenized.iloc[0][:12]))
print("Часть первой маски:", attention_mask[0, :25])
assert tokenizer.pad_token_id == 0
assert (padded[:, 0] == tokenizer.cls_token_id).all()
np.testing.assert_array_equal(attention_mask.sum(axis=1), tokenized.map(len))

Максимальная длина: 59
Форма padded: (2000, 59)
Первые токены: ['[CLS]', 'a', 'stirring', ',', 'funny', 'and', 'finally', 'transporting', 're', 'imagining', 'of', 'beauty']
Часть первой маски: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0]


### 4.3. Получение векторов `[CLS]`

Сначала явно показываем вычисление для 16 текстов. Размер выхода модели:
`(число текстов, число токенов, 768)`. Срез `[:, 0, :]` выбирает все тексты,
токен на позиции 0 (`[CLS]`) и все его 768 координат.

Для всех 2000 текстов функция из `lab.py` повторяет тот же расчёт порциями по 16
в режиме `torch.no_grad()` и сохраняет только `[CLS]`. Это снижает расход памяти.
Кэш проверяется по содержимому и порядку текстов, имени модели и способу извлечения.

In [9]:
input_ids = torch.tensor(padded, dtype=torch.long)
attention_mask_tensor = torch.tensor(attention_mask, dtype=torch.long)

with torch.no_grad():
    last_hidden_states = model(
        input_ids=input_ids[:16], attention_mask=attention_mask_tensor[:16]
    )
demo_features = last_hidden_states[0][:, 0, :].cpu().numpy().copy()
print("Скрытые состояния первого мини-батча:", tuple(last_hidden_states[0].shape))
print("Векторы [CLS] первого мини-батча:", demo_features.shape)
del last_hidden_states

features = lab.extract_bert_features(
    batch_1[0].tolist(), tokenizer=tokenizer, model=model,
    batch_size=16, device="cpu", threads=2
)
labels = batch_1[1]
print("Все признаки DistilBERT:", features.shape)
assert features.shape == (2000, 768) and np.isfinite(features).all()
np.testing.assert_allclose(features[:16], demo_features, atol=1e-5, rtol=1e-4)
del model
_ = gc.collect()

Скрытые состояния первого мини-батча: (16, 59, 768)
Векторы [CLS] первого мини-батча: (16, 768)
DistilBERT: проверенный кэш признаков (2000, 768)


Все признаки DistilBERT: (2000, 768)


### 4.4. Логистическая регрессия на признаках DistilBERT

In [10]:
# Те же индексы, что и у TF-IDF/GloVe: random_state=42, test_size=0.25.
train_features, test_features = features[train_idx], features[test_idx]
train_labels, test_labels = labels.iloc[train_idx], labels.iloc[test_idx]

lr_bert = LogisticRegression(max_iter=2000, random_state=42)
lr_bert.fit(train_features, train_labels)
remember_result("DistilBERT (Embeddings + LR)", lr_bert, train_features, test_features)

DistilBERT (Embeddings + LR) Train Accuracy: 0.9087
DistilBERT (Embeddings + LR) Test Accuracy:  0.8340
Количество признаков: 768


## Задание 5. Итоговый анализ

Сводная таблица строится из результатов выполненных ячеек. `gap` — разность
train и test accuracy; она помогает заметить, насколько качество на обучающих примерах
выше качества на новых. Сравнение относится к одному фиксированному разбиению;
разница в несколько верных ответов на 500 текстах не доказывает общее превосходство метода.

In [11]:
summary = lab.save_results(records, batch_1, train_idx, test_idx)
display(pd.DataFrame(records).rename(columns={
    "method": "Подход / модель", "train_accuracy": "Train Accuracy",
    "test_accuracy": "Test Accuracy", "gap": "Train − Test", "n_features": "Признаки"
}).round(4))
best = max(records, key=lambda item: item["test_accuracy"])
display(Markdown(
    f"**Лучший результат на этом разбиении: {best['method']} — "
    f"{best['test_accuracy']:.1%} Test Accuracy.** "
    f"Частый класс даёт {baseline:.1%}."
))

Результаты и отчёт сохранены в /home/salamat/Downloads/labka5/results


,Подход / модель,Train Accuracy,Test Accuracy,Train − Test,Признаки
0,TF-IDF (Unigrams),0.8327,0.674,0.1587,709
1,TF-IDF (N-Grams),0.8500,0.666,0.1840,1002
2,GloVe (Averaged),0.7073,0.714,-0.0067,25
3,DistilBERT (Embeddings + LR),0.9087,0.834,0.0747,768


**Лучший результат на этом разбиении: DistilBERT (Embeddings + LR) — 83.4% Test Accuracy.** Частый класс даёт 50.8%.

### Ответы на контрольные вопросы

Отчёт ниже сформирован по фактическим метрикам. Он также сохранён в `results/report.md`.

In [12]:
display(Markdown((lab.RESULTS_DIR / "report.md").read_text(encoding="utf-8")))

# Сравнительный анализ векторизации текста на SST-2

## 1. Данные и условия эксперимента

Использованы первые 2000 строк [SST-2](https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv).
Классы: 0 — негативный (959), 1 — позитивный (1041).
Разбиение: 1500 обучающих и 500 тестовых примеров,
`random_state=42`, без `stratify`. Позиционные индексы разбиения одинаковы у всех методов.
Словарь и IDF вычислены только по обучающей части. Предобученные GloVe и DistilBERT
не дообучались и не используют метки SST-2 при извлечении признаков.
Для каждого представления обучена `LogisticRegression(max_iter=2000, random_state=42)`.

Базовый классификатор всегда предсказывает наиболее частый **в обучающей части**
класс 1: train accuracy = 0.5247, test accuracy = 0.5080.

## 2–4. Представления текста

- **TF-IDF (Unigrams):** `min_df=5`, `ngram_range=(1, 1)`.
- **TF-IDF (N-Grams):** `min_df=5`, `ngram_range=(1, 2)`.
- **GloVe (Averaged):** `glove-twitter-25`, усреднение известных токенов после
  приведения к нижнему регистру и разделения регулярным выражением. Если известных
  токенов нет, используется нулевой вектор длины 25. Порядок слов не сохраняется.
- **DistilBERT (Embeddings + LR):** `distilbert-base-uncased`, ручные токенизация,
  padding и attention mask; 768 признаков из последнего скрытого слоя для `[CLS]`.
  Используются `eval()` и `torch.no_grad()`. Слишком длинные тексты усекаются до
  512 токенов; правый padding учитывается маской. Обработка небольшими батчами
  ограничивает расход памяти. Базовая модель специально не обучалась выдавать
  универсальные sentence embeddings; качество её `[CLS]` нельзя приравнивать к
  качеству специально обученных моделей Sentence Transformers или fine-tuning.

## 5. Реальные результаты запуска

| Подход / Модель | Train Accuracy | Test Accuracy | Train − Test | Признаков |
| --- | ---: | ---: | ---: | ---: |
| TF-IDF (Unigrams) | 0.8327 | 0.6740 | +0.1587 | 709 |
| TF-IDF (N-Grams) | 0.8500 | 0.6660 | +0.1840 | 1002 |
| GloVe (Averaged) | 0.7073 | 0.7140 | -0.0067 | 25 |
| DistilBERT (Embeddings + LR) | 0.9087 | 0.8340 | +0.0747 | 768 |

Переход от униграмм к униграммам и биграммам изменил test accuracy на -0.80 процентного пункта. Наибольший разрыв train − test у TF-IDF (N-Grams): 0.1840. Положительный разрыв указывает на возможное переобучение, но одной величины разрыва недостаточно для строгого вывода.

## Ответы на контрольные вопросы

**1. Какой метод показал наибольшую точность и почему?**

На этом разбиении лучший test accuracy равен **0.8340**:
**DistilBERT (Embeddings + LR)**. DistilBERT использует контекст и знания, полученные при предобучении на большом корпусе, поэтому может распознавать более сложные сочетания слов.
Это возможные объяснения результата, а не установленная причинная связь.
Разница лучшего результата с majority baseline равна
+32.60 процентного пункта.
На 500 тестовых объектах один правильный ответ меняет accuracy на
0.20 процентного пункта; один запуск не доказывает превосходство
метода на любых данных. Для устойчивого вывода нужны другие разбиения или кросс-валидация.

**2. Чем контекстуализированные эмбеддинги отличаются от статических?**

В GloVe и Word2Vec каждому слову соответствует один вектор независимо от предложения.
У BERT/DistilBERT представление токена зависит от окружающих токенов и позиции:
одно и то же слово в разных контекстах получает разные векторы. Усреднение GloVe
теряет порядок слов; self-attention с позиционными эмбеддингами учитывает контекст.

**3. Для чего нужен `attention_mask`?**

После дополнения предложений до общей длины padding не должен влиять на внимание.
Маска содержит 1 для настоящих и специальных токенов и 0 для padding. Модель
не использует позиции с маской 0 как источники информации при self-attention.
Маска не удаляет padding из массива и не гарантирует нулевые выходные векторы в
позициях padding; в этом задании берётся только вектор первого токена.

**4. Почему выбирается `[:, 0, :]`?**

Последний скрытый слой имеет форму `(число текстов, число токенов, 768)`.
Первая ось `:` выбирает все тексты, индекс `0` — первый специальный токен `[CLS]`,
последняя ось `:` — все его 768 признаков. После слоёв self-attention этот токен
содержит контекстную информацию о последовательности и даёт фиксированный по длине
вектор для LR. Это соглашение из задания; у базового DistilBERT нет отдельного
обученного sentence-pooling слоя, и `[CLS]` не гарантирует оптимальный вектор текста.

## Воспроизводимость

Запуск: `venv/bin/python lab.py`. Таблица сохранена в `results/metrics.csv`,
метрики с параметрами разбиения и базовым уровнем — в `results/metrics.json`.
Кэш CLS проверяется по SHA-256 текстов с учётом их порядка, имени модели и способу
извлечения. Для принудительного пересчёта используйте `--refresh-features`.

## Документация

- [DistilBERT: входы, attention mask и выходы скрытых слоёв](https://huggingface.co/docs/transformers/model_doc/distilbert).
- [Gensim Downloader: загрузка предобученных векторов](https://radimrehurek.com/gensim/downloader.html).


### Источники

- [SST-2: файл из задания](https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv).
- [Gensim Downloader: загрузка предобученных векторов](https://radimrehurek.com/gensim/downloader.html).
- [Hugging Face: DistilBERT, входы и скрытые состояния](https://huggingface.co/docs/transformers/model_doc/distilbert).

Обучение TF-IDF выполнялось только на train. Для выбора гиперпараметров test не использовался.
Базовый `[CLS]` DistilBERT не специально обучен как универсальный вектор предложения;
здесь он используется как набор признаков согласно заданию.